# aufbau — type-system workbench

The low level, exposed. A **type system** is a tuple `(Σ, L, R, Θ)`: a signature `Σ`
(the grammar's type productions), a leaf algebra `L`, a rewrite theory `R` (the `⇝`
rules), and typing rules `Θ`. A **type** is a term `τ ∈ T(Σ, V)`. Everything below
operates on those objects directly: parse, unify, normalize, compile to IR, check.

In [ ]:
import aufbau
print([c for c in dir(aufbau) if c[:1].isupper()])

## 1 · Σ — the signature is the grammar
`signature()` lists every constructor with an arity it occurs at; nullary = a leaf.

In [ ]:
STLC = r"""
Identifier  ::= /[a-z]+/
TypeName    ::= /[A-Z][a-zA-Z0-9]*/
AtomicType  ::= TypeName | '(' Type ')'
FunctionType::= AtomicType '->' Type
Type        ::= AtomicType | FunctionType

Variable(var)    ::= Identifier[x]
Lambda(lambda)   ::= 'λ' Identifier[a] ':' Type[τ] '.' Expr[e]
Application(app) ::= Expr[l] Atom[r]
Atom             ::= Variable | '(' Expr ')'
Expr             ::= Atom | Lambda | Application

x ∈ Γ
----------- (var)
Γ(x)

Γ[a:τ] ⊢ e : ?B
--------------- (lambda)
τ -> ?B

Γ ⊢ l : ?A -> ?B, Γ ⊢ r : ?A
---------------------------- (app)
?B
"""
g = aufbau.SPG(STLC)
print('start    :', g.start)
print('Σ        :', g.signature())
print('rules Θ  :', g.rule_names())
print('rewrites :', g.rewrites())

## 2 · A type is a term `T(Σ, V)`
`parse_type` lifts surface syntax to a tree; `show` renders it back. `?A` is a
variable (`V`), a name is a leaf (`L`), a production is a constructor.

In [ ]:
def tree(t, d=0):
    head = t.label() or str(t)
    print('  '*d + head)
    for k in t.children(): tree(k, d+1)

t = g.parse_type('(A -> B) -> C')
tree(t)
print('surface :', g.show(t))
print('ground? :', t.is_ground(), '| var?', g.parse_type('?X').is_var())

## 3 · Unification — the one checking primitive
Free first-order unification on Σ-terms. A variable captures a whole **subtree**
(impossible in the old leftmost-string model). A constructor/leaf clash fails.

In [ ]:
print('?A -> ?B   ~  (A -> B) -> C :', g.unify('?A -> ?B', '(A -> B) -> C'))
print('A -> ?X    ~  A -> (B -> C) :', g.unify('A -> ?X', 'A -> (B -> C)'))
print('A -> B     ~  A -> C        :', g.unify('A -> B', 'A -> C'))

## 4 · Normalization — the equational theory `R`
`R` is the `⇝` rules. `unify_modulo = unify ∘ normalize`. With `Bool ⇝ Unit + Unit`,
`Bool` and `Unit + Unit` are the *same* type; free unification alone clashes.

In [ ]:
SUMS = r"""
TypeName   ::= /[A-Z][a-z]*/
AtomicType ::= TypeName | '(' Type ')'
SumType    ::= AtomicType '+' Type
Type       ::= AtomicType | SumType

Word       ::= 'x' | 'y'
Annot(annot) ::= '(' Word[w] ':' Type[τ] ')'
Same(same)   ::= Annot[a] '~' Annot[b]
Expr         ::= Annot | Same

----------- (annot)
τ

Γ ⊢ a : ?T, Γ ⊢ b : ?T
---------------------- (same)
?T

Bool ⇝ Unit + Unit
"""
sums = aufbau.SPG(SUMS)
print('normalize Bool   :', sums.show(sums.normalize('Bool')))
print('unify        Bool ~ Unit+Unit :', sums.unify('Bool', 'Unit + Unit'))
print('unify_modulo Bool ~ Unit+Unit :', sums.unify_modulo('Bool', 'Unit + Unit'))

## 5 · Rules compile to an IR
A typing rule is sugar. `ir(name)` shows the instruction stream it lowers to —
each op is one primitive call (`eval`, `ascribe`=unify_modulo, `emit`).

In [ ]:
for r in ['var', 'lambda', 'app']:
    print(g.ir(r))

## 6 · Type-checking, end to end
`Synthesizer` parses + checks. Context types are parsed into trees, so `A -> B`
unifies structurally against the `app` rule's `?A -> ?B`.

In [ ]:
s = aufbau.Synthesizer(STLC, 'f x')
s.add_to_ctx('f', 'A -> B')
s.add_to_ctx('x', 'A')
ast = s.ast()
for root in ast.roots:
    print('type:', ast.type_of(root.evidence))

In [ ]:
# the sums system: the rewrite is load-bearing here
s = aufbau.Synthesizer(SUMS, '(x : Bool) ~ (y : Unit + Unit)')
ast = s.ast()
for root in ast.roots:
    print('type:', ast.type_of(root.evidence))   # SumType(Unit, Unit) — the normal form

## 7 · Build your own
Edit `MY` — add constructors (productions), equations (`⇝`), rules (`Θ`) — and
probe with `parse_type / unify / unify_modulo / normalize / ir`. This is the loop.

In [ ]:
MY = r"""
Name ::= /[A-Z][a-z]*/
Atom ::= Name | '(' Type ')'
Prod ::= Atom '*' Type
Type ::= Atom | Prod

Pair ⇝ Fst * Snd
"""
my = aufbau.SPG(MY)
print('Σ        :', my.signature())
print('parse    :', my.show(my.parse_type('A * (B * C)')))
print('unify    :', my.unify('?X * ?Y', 'A * (B * C)'))
print('normalize:', my.show(my.normalize('Pair')))